In [ ]:
%%sql -r dataframe_12
USE ROLE CHEETAH_DATA5035_ROLE

# MOAB Rover Survey Lab: Feature Engineering in Snowflake

## Goal
In this lab, we will engineer new features from rover-collected survey data using:
- Python UDFs
- SQL views
- AI prompts

## Raw Inputs
- Easting
- Northing
- Sensor measurement

## Engineered Features
1. Grid tile, survey unit, and subcell assignment
2. Tile-level aggregated measurement signals
3. Comparison to normal range
4. Remediation prioritization

## SCALARS Mapping
- **Simplify**: convert coordinates into tile IDs
- **Aggregate**: summarize measurements at the tile level using multi-level aggregation
- **Assess**: compare readings to expected range
- **Rank / Score**: prioritize areas for remediation

In [ ]:
%%sql -r source_data
-- Change to your user's schema
USE SCHEMA data5035.CHEETAH;
SELECT * FROM data5035.spring26.sdg_001_ra226_scandata limit 10;

## Convert from Coordinates to Grid

The input data is provided in directional distances on a flat map projection. Northing and Easting indicate how far to go in those directions (up and right) in US Feet relative to a known starting point. Our purpose here is to map those directional distances on to a three-level grid.

### Measurement
* **Tiles** are the largest areas. They are composed of a grid of **Survey Units** 21 tiles wide x 18 tiles tall.
* **Survey Units** measure 32.81 ft x 32.81 ft square
* **Subcells** are square subdivisions within the **Survey Units** laid out 10x10

### Labeling
* **Tiles** are coded by row letter and column letter starting at AA in the bottom-left of our map, given some known origin (2180160.001, 6660000.000). AA indicates 1st row, 1st column. AB indicates 1st row, 2nd column to the left. BA indicates 2nd row up, 1st column.
* Within each Tile, **Survey Units** are numbered starting in the top-left corner, proceeeding right, then down to the beginning of the next row (as if you're reading down a page)
* Within each Survey Unit, **Subcells** are numbered starting in the bottom-left corner, proceeduing right, then up to the beginning of thext row (as if you're reading from the bottom of a page up)

**Create a Python UDF to convert x (easting), y (northing) into the grid labels.**

### Testing

```
    >>> convert_xy(2180160.0001, 6660000.0000)  
    ('AA', 358, 1)

    >>> convert_xy(2180160.0001 + 32.81*21.01, 6660000.0000)
    ('AB', 358, 1)

    >>> convert_xy(2180160.0001 + 32.81*22.01 + 4, 6660000.0000 + 32.81*19.01 + 4)
    ('BB', 338, 12)
```

In [ ]:
CREATE OR REPLACE FUNCTION CONVERT_XY(
        X FLOAT,
        Y FLOAT,
        ORIGIN_X FLOAT,
        ORIGIN_Y FLOAT,
        SU_SIZE FLOAT,
        TILE_GRID_X NUMBER(38,0),
        TILE_GRID_Y NUMBER(38,0),
        SUBCELL_GRID NUMBER(38,0)
    )
    RETURNS OBJECT
    LANGUAGE PYTHON
    RUNTIME_VERSION = '3.11'
    HANDLER = 'convert_xy'
    AS 
    $$
def convert_xy(x, y, origin_x, origin_y, su_size, tile_grid_x, tile_grid_y, subcell_grid):

    # Offset from origin (bottom-left of tile AA)
    dx = x - origin_x
    dy = y - origin_y

    if dx < 0 or dy < 0:
        return {"tile": "BAD_DATA", "survey_unit": None, "sub_cell": None,
                "error": f"Negative offset: dx={dx:.4f}, dy={dy:.4f}"}

    # Tile indices (0-indexed: 0=A, 1=B, ...)
    tile_width  = su_size * tile_grid_x
    tile_height = su_size * tile_grid_y

    tile_col_idx = int(dx / tile_width)
    tile_row_idx = int(dy / tile_height)

    if tile_col_idx > 25 or tile_row_idx > 25:
        return {"tile": "BAD_DATA", "survey_unit": None, "sub_cell": None,
                "error": f"Tile index exceeds alphabet (col={tile_col_idx}, row={tile_row_idx})"}

    # First letter = row (northing), second = column (easting)
    tile_code = chr(ord('A') + tile_row_idx) + chr(ord('A') + tile_col_idx)

    # Position within the tile
    x_in_tile = dx - tile_col_idx * tile_width
    y_in_tile = dy - tile_row_idx * tile_height

    # Survey Unit: numbered top-left=1, right then down
    su_col          = min(int(x_in_tile / su_size), tile_grid_x - 1)
    su_row_from_bot = min(int(y_in_tile / su_size), tile_grid_y - 1)
    su_row_from_top = (tile_grid_y - 1) - su_row_from_bot
    su_number       = su_row_from_top * tile_grid_x + su_col + 1

    # Position within the Survey Unit
    x_in_su = x_in_tile - su_col          * su_size
    y_in_su = y_in_tile - su_row_from_bot * su_size

    # Sub-cell: numbered bottom-left=1, right then up
    sc_size         = su_size / subcell_grid
    sc_col          = min(int(x_in_su / sc_size), subcell_grid - 1)
    sc_row_from_bot = min(int(y_in_su / sc_size), subcell_grid - 1)
    sc_number       = sc_row_from_bot * subcell_grid + sc_col + 1

    return {"tile": tile_code, "survey_unit": su_number, "sub_cell": sc_number}
    $$;

In [ ]:
%%sql -r dataframe_3
    CREATE OR REPLACE FUNCTION CONVERT_XY(X FLOAT, Y FLOAT)
    RETURNS OBJECT
    LANGUAGE SQL
    AS
    $$
        SELECT CONVERT_XY(
            X, Y,
            2180160.0001,
            6660000.0000,
            32.81,
            21,
            18,
            10
        )
    $$;

In [ ]:
%%sql -r dataframe_4
select convert_xy(2180160.0001, 6660000.0000);

In [ ]:
%%sql -r dataframe_5
select convert_xy(2180160.0001 + 32.81*21.01, 6660000.0000);

In [ ]:
%%sql -r dataframe_6
select convert_xy(2180160.0001 + 32.81*22.01 + 4, 6660000.0000 + 32.81*19.01 + 4)

In [ ]:
%%sql -r dataframe_7
SELECT 
    convert_xy(easting, northing) AS coordinates, 
    coordinates:su::INTEGER AS su,
    coordinates:subcell::INTEGER AS subcell,
    coordinates:tile::STRING AS tile,
    * 
FROM 
    data5035.spring26.sdg_001_ra226_scandata 
LIMIT 100;

## Compute Layered Averages

In [ ]:
SELECT 
    convert_xy(easting, northing) AS coordinates, 
    coordinates:tile::STRING      AS tile,
    coordinates:survey_unit::INT  AS survey_unit,
    coordinates:sub_cell::INT     AS sub_cell,
    avg(reading) 
FROM 
    data5035.spring26.sdg_001_ra226_scandata 
GROUP BY ALL
ORDER BY 2, 3, 4

In [ ]:
-- =============================================================================
-- LAYERED AVERAGES  —  Radiation Site Survey
-- Each level treats its children equally (weight = 1 per child),
-- regardless of how many raw samples were collected there.
-- =============================================================================

WITH

-- Layer 0: attach grid labels to every raw reading
labeled AS (
    SELECT
        r.READING,
        loc:tile::VARCHAR    AS TILE,
        loc:survey_unit::INT AS SURVEY_UNIT,
        loc:sub_cell::INT    AS SUB_CELL
    FROM DATA5035.SPRING26.SDG_001_RA226_SCANDATA r,
         LATERAL (SELECT CONVERT_XY(r.EASTING, r.NORTHING) AS loc) l
    WHERE loc:tile::VARCHAR <> 'BAD_DATA'
),

-- Layer 1: one average per sub-cell
--   N raw samples in the same square meter → treated as a single value
subcell_avgs AS (
    SELECT
        TILE,
        SURVEY_UNIT,
        SUB_CELL,
        AVG(READING) AS AVG_READING
    FROM labeled
    GROUP BY TILE, SURVEY_UNIT, SUB_CELL
),

-- Layer 2: one average per survey unit  (average of sub-cell averages)
--   Each sub-cell gets weight 1 regardless of its sample count
su_avgs AS (
    SELECT
        TILE,
        SURVEY_UNIT,
        AVG(AVG_READING) AS AVG_READING
    FROM subcell_avgs
    GROUP BY TILE, SURVEY_UNIT
),

-- Layer 3: one average per tile  (average of survey unit averages)
--   Each survey unit gets weight 1 regardless of its sub-cell count
tile_avgs AS (
    SELECT
        TILE,
        AVG(AVG_READING) AS AVG_READING
    FROM su_avgs
    GROUP BY TILE
)

-- Final output — pull whichever level(s) you need:
SELECT * FROM tile_avgs       ORDER BY TILE

In [ ]:
-- Final output — all three levels side by side
WITH
labeled AS (
    SELECT
        r.READING,
        loc:tile::VARCHAR    AS TILE,
        loc:survey_unit::INT AS SURVEY_UNIT,
        loc:sub_cell::INT    AS SUB_CELL
    FROM DATA5035.SPRING26.SDG_001_RA226_SCANDATA r,
         LATERAL (SELECT CONVERT_XY(r.EASTING, r.NORTHING) AS loc) l
    WHERE loc:tile::VARCHAR <> 'BAD_DATA'
),
subcell_avgs AS (
    SELECT
        TILE,
        SURVEY_UNIT,
        SUB_CELL,
        AVG(READING) AS AVG_READING
    FROM labeled
    GROUP BY TILE, SURVEY_UNIT, SUB_CELL
),
su_avgs AS (
    SELECT
        TILE,
        SURVEY_UNIT,
        AVG(AVG_READING) AS AVG_READING
    FROM subcell_avgs
    GROUP BY TILE, SURVEY_UNIT
),
tile_avgs AS (
    SELECT
        TILE,
        AVG(AVG_READING) AS AVG_READING
    FROM su_avgs
    GROUP BY TILE
)
SELECT
    s.TILE,
    s.SURVEY_UNIT,
    s.SUB_CELL,
    ROUND(s.AVG_READING,  4) AS AVG_SUBCELL,
    ROUND(su.AVG_READING, 4) AS AVG_SU,
    ROUND(t.AVG_READING,  4) AS AVG_TILE
FROM subcell_avgs s
JOIN su_avgs  su ON s.TILE = su.TILE AND s.SURVEY_UNIT = su.SURVEY_UNIT
JOIN tile_avgs t ON s.TILE = t.TILE
ORDER BY s.TILE, s.SURVEY_UNIT, s.SUB_CELL

## Compare to Reference Ranges

This measurement is of Radium-226 levels.
* `<5` - OK
* `5 <= X < 7.4` - Warning
* `>= 7.4` - Alarm

In [ ]:
%%sql -r dataframe_9


## Prioritize

Once we've build everything out, let's use AI to make some recommendations on remediation priorities.

In [ ]:
%%sql -r dataframe_10
